# Fraud monitoring

During my bank internship, I came across situations involving fraudulent transfers. This project uses the public PaySim dataset to explore payment types, amounts and daily changes. It does not use bank or customer records.

The data are simulated. `isFraud` is an existing label, so the analysis looks back at recorded activity. Download instructions are in [data/README.md](data/README.md).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

path = Path("data/PS_20174392719_1491204439457_log.csv")
if not path.exists():
    raise FileNotFoundError("Download the PaySim CSV first; see data/README.md.")

data = pd.read_csv(path, usecols=["step", "type", "amount", "isFraud"],
                   dtype={"type": "category"})
display(data.head())
print(f"Transactions: {len(data):,}")


## 1. Check the data

Start with missing values and a few basic checks. `isFraud` is 0 or 1, so its mean gives the proportion of transactions labelled fraud.

In [ ]:
display(data.isna().sum().rename("Missing values"))
assert not data.isna().any().any(), "Check missing values before continuing."
assert data["isFraud"].isin([0, 1]).all(), "Labels must be 0 or 1."
assert np.isfinite(data[["step", "amount"]]).all().all()
assert data["amount"].ge(0).all()
assert data["step"].ge(1).all() and data["step"].eq(np.floor(data["step"])).all()
print(f"Overall fraud rate: {data['isFraud'].mean():.3%}")

## 2. Compare payment types

Count transactions and fraud records within each payment type. Divide fraud count by transaction count to get the rate for that type.

In [ ]:
by_type = data.groupby("type", observed=True)["isFraud"].agg(
    transactions="size", fraud_count="sum", fraud_rate="mean")
by_type["fraud_percent"] = 100 * by_type["fraud_rate"]
display(by_type[["transactions", "fraud_count", "fraud_percent"]].round(3))

by_type["fraud_percent"].plot.bar(rot=0, figsize=(8, 4))
plt.ylabel("Fraud transactions (%)")
plt.xlabel("Payment type")
plt.title("Fraud rate within each payment type")
plt.tight_layout()
plt.show()

## 3. Look at transaction amounts

Compare the mean and median across all payment types. A few large transactions can pull the mean upwards.

The histogram uses log(1 + amount) to show the long tail more clearly. Each group is normalised separately, so the chart compares the shapes of the distributions. Amounts are in dataset currency units.

In [ ]:
display(data.groupby("isFraud")["amount"].agg(["count", "mean", "median"]).round(2))

bins = np.linspace(0, np.log1p(data["amount"].max()) + 0.01, 35)
for label, name in [(0, "Non-fraud"), (1, "Fraud")]:
    amounts = data.loc[data["isFraud"] == label, "amount"]
    plt.hist(np.log1p(amounts), bins=bins, density=True, alpha=0.5, label=name)
plt.xlabel("log(1 + amount in dataset currency units)")
plt.ylabel("Density within each group")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Daily changes

Each `step` represents one simulated hour. Group steps 1–24 into day 1, 25–48 into day 2, and so on.

Track daily transaction counts, amounts and fraud rates. Keep counts next to rates: the fraud percentage can increase simply because there are fewer non-fraud transactions.

The table includes all observed days. The charts use days with 24 observed hours so that partial days do not look like sudden drops in activity. Missing hours could mean no activity or missing records; the data do not tell us which.

In [ ]:
data["day"] = ((data["step"] - 1) // 24 + 1).astype(int)
daily = data.groupby("day").agg(
    observed_hours=("step", "nunique"),
    transactions=("isFraud", "size"),
    total_amount=("amount", "sum"),
    mean_amount=("amount", "mean"),
    median_amount=("amount", "median"),
    fraud_count=("isFraud", "sum")
)
daily["fraud_percent"] = 100 * daily["fraud_count"] / daily["transactions"]
display(daily.round(3))

complete_days = daily.loc[daily["observed_hours"] == 24]
print(f"Days with all 24 hours represented: {len(complete_days)}")
if len(complete_days) < len(daily):
    print("Some days have fewer observed hours; compare their totals carefully.")

if complete_days.empty:
    print("No complete days available for the trend charts. Inspect the daily table instead.")
else:
    fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
    # Keep gaps for excluded days rather than connecting across them.
    plotted = complete_days.reindex(range(int(daily.index.min()), int(daily.index.max()) + 1))
    plotted["transactions"].plot(ax=axes[0], marker="o")
    axes[0].set_ylabel("Transactions")
    plotted["fraud_percent"].plot(ax=axes[1], marker="o", color="darkorange")
    axes[1].set_ylabel("Fraud (%)")
    plotted[["mean_amount", "median_amount"]].plot(ax=axes[2], marker="o")
    axes[2].set_ylabel("Amount (dataset units)")
    axes[2].set_xlabel("Simulation day")
    fig.suptitle("Daily payment data: days with all 24 hours represented")
    fig.tight_layout()
    plt.show()


## 5. Look more closely at one day

Choose the complete day with the highest fraud rate and break it down by payment type. If days tie, use the first one.

Compare this table with section 2. Is the high rate accompanied by more fraud records, a smaller number of transactions, or a different mix of payment types? This is a starting point for investigation, not a formal anomaly test.

In [ ]:
if not complete_days.empty:
    focus_day = complete_days["fraud_percent"].idxmax()
    print(f"Day selected for inspection: {focus_day}")
    display(daily.loc[[focus_day]].round(3))
    focus = data.loc[data["day"] == focus_day]
    focus_by_type = focus.groupby("type", observed=True)["isFraud"].agg(
        transactions="size", fraud_count="sum", fraud_rate="mean")
    focus_by_type["fraud_percent"] = 100 * focus_by_type["fraud_rate"]
    display(focus_by_type[["transactions", "fraud_count", "fraud_percent"]].round(3))
else:
    print("No complete day to select; review observed hours in the daily table.")


## Notes

The notebook has not yet been run on the full CSV in this revision. Add observations here after running it: which payment types differ, how daily counts and rates move together, and what stands out about the selected day.

These are simulated transactions. A high amount or rate alone does not explain why it occurred. Transaction amounts are not final losses, and the fraud labels would not necessarily be available immediately in practice.